In [ ]:
# %%
import pickle
import pandas as pd
from IPython.display import display, Markdown

# ==========================================
# ⏱️ 量化时光机：机构级每日战报
# ==========================================
TARGET_DATE = "2026-04-21" 
report_file = f"Daily_Reports/Report_{TARGET_DATE}.pkl"

try:
    with open(report_file, 'rb') as f:
        snap = pickle.load(f)
        
    # 标题头
    display(Markdown(f"# 📊 宽客实战日报 | 交易日: {snap['date']}"))
    display(Markdown(f"### 👑 今日市场绝对主线 (King Factor): **`{snap['king_factor']}`**"))
    display(Markdown("---"))
    
    # 模块 1：OLS 市场风向归因
    r2_fac = snap['ols_report']['r2_factors'] * 100
    r2_ind = snap['ols_report']['r2_ind'] * 100
    display(Markdown(f"#### 📡 1. 市场微观结构探测 (解释力 - 因子: `{r2_fac:.2f}%` | 行业: `{r2_ind:.2f}%`)"))
    
    # 对 Beta 和 T 值进行高亮上色
    style_ols = snap['ols_report']['stats_df'].style.applymap(
        lambda x: 'color: red; font-weight: bold' if isinstance(x, str) and '追捧 ★' in x else 
                  ('color: green; font-weight: bold' if isinstance(x, str) and '抛售 ★' in x else ''),
        subset=['风向状态']
    ).background_gradient(subset=['Beta系数'], cmap='coolwarm', vmin=-0.05, vmax=0.05)
    display(style_ols)
    
    # 模块 2：强势板块与龙头画像
    display(Markdown("#### 🎯 2. 强势板块下钻与龙头画像"))
    # 横向展示领涨和领跌板块
    col1, col2 = snap['drill_report']['top_industries'], snap['drill_report']['bottom_industries']
    display(pd.concat([col1.set_index('板块名称'), col2.set_index('板块名称')], axis=1, keys=['🔥 领涨板块', '❄️ 领跌板块']))
    
    display(Markdown("**【强势板块内部资金画像】**"))
    display(snap['drill_report']['portraits_df'].style.set_properties(**{'text-align': 'left'}))
    display(Markdown("---"))

    # 模块 3：明日交易底仓狙击池
    display(Markdown(f"#### 🔫 3. 明日开盘狙击池 (按核心因子 **`{snap['king_factor']}`** 降序排列)"))
    top_10_picks = snap['top_picks'].head(10)[['code', 'name', snap['king_factor'], 'Momentum', 'Liquidity', 'Size']]
    display(top_10_picks.style.background_gradient(subset=[snap['king_factor']], cmap='YlOrRd'))

except FileNotFoundError:
    print(f"❌ 找不到 {TARGET_DATE} 的战报！")
# %%